# Chapter - 2: Coding attention mechanism

## 2.1 Attending to different parts of the i/p with self attention

### 2.2.1 A simple self-attention mechanism without trainabe weights

In [4]:
#Simple EG:
import torch

inputs = torch.tensor(
    [[0.43,0.15,0.89],# Your  (x^1)
     [0.55,0.87,0.66],# journey (x^2)
     [0.57,0.85,0.64],# starts (x^3)
     [0.22,0.58,0.33],# with (x^4)
     [0.77,0.25,0.10],# one (x^5)
     [0.05,0.80,0.55] # step (x^6)
    ]
)
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [5]:
input_query = inputs[1]                       #2nd vector in inputs
input_query

tensor([0.5500, 0.8700, 0.6600])

In [6]:
print("Hi")

Hi


In [7]:
inp_1 = inputs[0]     #1st vector in inputs
inp_1

tensor([0.4300, 0.1500, 0.8900])

In [8]:
0.55*0.43 + 0.87*0.15 + 0.66*0.89           #dot product of input_query and inp_1

0.9544

In [9]:
torch.dot(input_query,inp_1)                #using pyTorch

tensor(0.9544)

In [10]:
res = 0.

for idx , ele in enumerate(inputs[0]):
    res += inputs[0][idx] * input_query[idx]

res

tensor(0.9544)

In [11]:
i = 0
res = torch.dot(inputs[i],input_query)
res

tensor(0.9544)

In [12]:
torch.empty(inputs.shape[0])

tensor([0., 0., 0., 0., 0., 0.])

In [13]:
query = inputs[1]           #2nd i/p token is the query

attn_score_2 = torch.empty(inputs.shape[0])     # we are creating a empty tensor (vector) with 0s with the size of  the input vectors, i.e here 6, for efficient memory management     

for i, x_i in enumerate(inputs):
    attn_score_2[i] = torch.dot(x_i,query)  # dot product 

print(attn_score_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [14]:
# Normalizing each tensors  above
attn_weights_2_tmp = attn_score_2/attn_score_2.sum()
attn_weights_2_tmp

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [15]:
# we are normalizing to make the summ of those vctors to be equal to 1
attn_weights_2_tmp.sum()

tensor(1.0000)

In [16]:
#Another way of normalization is by using sofmax
def softmax_native(x):                              #self implemented softmax
    return torch.exp(x)/torch.exp(x).sum()
softmax_native(attn_score_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [17]:
# In practical use, the torch version of softmax normalization, in some cases, the native defined f() might fail. 
attn_weights_2 = torch.softmax(attn_score_2,dim=0)
attn_weights_2

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [18]:
query = inputs[1]

context_vect_2 = torch.zeros(query.shape)

for i, x_i in enumerate(inputs):
    
    print(f"{attn_weights_2[i]} -----> {inputs[i]}")                       #just a visualization
    context_vect_2 += attn_weights_2[i] * inputs[i]

print(context_vect_2)

0.13854756951332092 -----> tensor([0.4300, 0.1500, 0.8900])
0.2378913015127182 -----> tensor([0.5500, 0.8700, 0.6600])
0.23327402770519257 -----> tensor([0.5700, 0.8500, 0.6400])
0.12399158626794815 -----> tensor([0.2200, 0.5800, 0.3300])
0.10818186402320862 -----> tensor([0.7700, 0.2500, 0.1000])
0.15811361372470856 -----> tensor([0.0500, 0.8000, 0.5500])
tensor([0.4419, 0.6515, 0.5683])


# 2.2.2 A simple self-attention mechanism without trainable weights

In [20]:
attn_score = torch.empty(6,6)                   

for i, x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        attn_score[i,j] = torch.dot(x_i,x_j)  

print(attn_score)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [21]:
# In PyTorch, there is an efficient way of doing matrix multiplication, and that is as follows:

attn_score = inputs @ inputs.T
attn_score

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [22]:
# Normalizing the attn_score by softmax:

attn_weight = torch.softmax(attn_score,dim = 1)              #what the dim = 1 ----> means each of the rown sum equals to 1 (normalize)
attn_weight

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [23]:
# dim means:

attn_weight.sum(dim = 1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [24]:
# finding context vector for all the attention weights:

all_context_vector = attn_weight @ inputs
all_context_vector

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [25]:
# cleaned version of everything done in this session:

attn_score = inputs @ inputs.T
attn_weight = torch.softmax(attn_score,dim = 1) 

all_context_vector = attn_weight @ inputs
all_context_vector


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

# Implemting self-attention with trainable weights

## computing attention weights step-by-step

In [28]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [29]:
x_2 = inputs[1]
d_in = inputs.shape[1]              # dimension of the input <---- here its three(3)
d_out = 2                           #dimension of output    (output embedding size)

In [30]:
torch.manual_seed(123)

W_query =torch.nn.Parameter(torch.rand(d_in, d_out))                # torch.Parameter is like a wrapper around ot to make it trainable   
W_key =torch.nn.Parameter(torch.rand(d_in, d_out))
W_value =torch.nn.Parameter(torch.rand(d_in, d_out))

In [31]:
W_query

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True)

In [32]:
W_key

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]], requires_grad=True)

In [33]:
W_value

Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]], requires_grad=True)

In [34]:
query_2 = x_2 @ W_query
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [35]:
keys = inputs @ W_key
value = inputs @ W_value

In [36]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)

In [37]:
value

tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]], grad_fn=<MmBackward0>)

In [38]:
key_2 = keys[1]
attn_scor22 = torch.dot(key_2,query_2) 
attn_scor22

tensor(1.8524, grad_fn=<DotBackward0>)

In [39]:
attn_score_2 = query_2 @ keys.T
attn_score_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)

In [40]:
dim_k = keys.shape[1]

attn_weights_2 = torch.softmax(attn_score_2 / dim_k ** 0.5 , dim = -1)                        # dim_k ** 0.5 ----> sqrt(dim_k)

attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [41]:
context_vect_2 = attn_weights_2 @ value
context_vect_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

# Implementing a compact Selfattention Class

In [43]:
import torch.nn as nn

class selfAttention_v1(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        self.W_query =torch.nn.Parameter(torch.rand(d_in, d_out))                # torch.Parameter is like a wrapper around ot to make it trainable   
        self.W_key =torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_value =torch.nn.Parameter(torch.rand(d_in, d_out))


    def forward(self,x):
        queries = inputs @ W_query
        keys = inputs @ W_key
        values = inputs @ W_value

        attn_score = queries @ keys.T
        attn_weights = torch.softmax(attn_score / dim_k ** 0.5 , dim = -1)                        # dim_k ** 0.5 ----> sqrt(dim_k)
        contxt_vec = attn_weights @ values
        return contxt_vec

torch.manual_seed(123)
sa_v1 = selfAttention_v1(d_in,d_out)
sa_v1(inputs)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [44]:
import torch.nn as nn

class selfAttention_v2(nn.Module):
    def __init__(self,d_in,d_out, qkv_bias = False):
        super().__init__()
        self.W_query =torch.nn.Linear(d_in, d_out , bias = qkv_bias)                #torch.nn.Linear is a slightly optimal version or an implementation of  torch.Parameter 
        self.W_key =torch.nn.Linear(d_in, d_out , bias = qkv_bias)  
        self.W_value =torch.nn.Linear(d_in, d_out , bias = qkv_bias)  


    def forward(self,x):
        queries = self.W_query(inputs)
        keys = self.W_keys(inputs)
        values = self.W_value(inputs)

        attn_score = queries @ keys.T
        attn_weights = torch.softmax(attn_score / dim_k ** 0.5 , dim = -1)                        # dim_k ** 0.5 ----> sqrt(dim_k)
        contxt_vec = attn_weights @ values
        return contxt_vec

torch.manual_seed(789)
sa_v2 = selfAttention_v1(d_in,d_out)
sa_v2(inputs)                                                             #The diff in o/p is cause of the variation in weight distribtion

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

# Hiding future words with casual attention

## Applying a casual attention mask

In [47]:
torch.manual_seed(789)

sa_v2 = selfAttention_v2(d_in, d_out)

queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
values = sa_v2.W_value(inputs)

attn_score = queries @ keys.T
attn_weights = torch.softmax(attn_score / dim_k ** 0.5 , dim = -1)

In [48]:
attn_weights

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

In [49]:
# Masking:

context_len = attn_score.shape[0]
mask_simpe = torch.tril(torch.ones(context_len,context_len))
print(mask_simpe)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [50]:
masked_simple =attn_weights * mask_simpe
masked_simple                                              # in the output the um of rows dont add upto 1, so we need to optimize or normalize it

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)

In [51]:
row_sums = masked_simple.sum(dim = -1, keepdim =True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [52]:
#Another way of masking it is :
mask = torch.triu(torch.ones(context_len,context_len),diagonal = 1)
maksed = attn_score.masked_fill(mask.bool(),-torch.inf)
print(maksed)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


## masking additional attention weights with dropouts

In [54]:
torch.manual_seed(123)

layer = torch.nn.Dropout(0.5)                 #0.5 refers to 50% of the part is being masked in the above masked

In [55]:
eg = torch.ones(6,6)
eg

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [56]:
layer(eg)

tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])

In [57]:
layer(masked_simple_norm)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.0000, 0.4638, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.3968, 0.3775, 0.3941, 0.0000],
        [0.3869, 0.3327, 0.0000, 0.0000, 0.3331, 0.3058]],
       grad_fn=<MulBackward0>)

## Implementing a compact casual self-attention class

In [59]:
batch = torch.stack((inputs,inputs),dim = 0)
batch.shape                                        # 2 i/p , 6 tokens in each i/p tensor , 3 dim for each tokens

torch.Size([2, 6, 3])

In [60]:
import torch.nn as nn

class CasualAttention(nn.Module):
    def __init__(self,d_in,d_out,context_len, dropout, qkv_bias = False):
        super().__init__()
        self.W_query =torch.nn.Linear(d_in, d_out , bias = qkv_bias)               
        self.W_key =torch.nn.Linear(d_in, d_out , bias = qkv_bias)  
        self.W_value =torch.nn.Linear(d_in, d_out , bias = qkv_bias)  
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer("mask",torch.triu(torch.ones(context_len,context_len),diagonal = 1))
    
    def forward(self,x):
        b,num_tokens,d_in = x.shape                        # x = batch ---> 2 x 6 x 3
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2)           # changed transpose
        attn_scores.masked_fill_(                             # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens],-torch.inf                        #
        )
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5 , dim = -1
        )
        attn_weights = self.dropout(attn_weights)              # New
        contxt_vec = attn_weights @ values

        
        return contxt_vec

torch.manual_seed(789)
context_len = batch.shape[1]                                               # batch.shape ---> torch.Size([2, 6, 3]) and we need 6
dropout = 0.0
ca = CasualAttention(d_in, d_out ,context_len, dropout)

ca(batch)                                                            

tensor([[[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]],

        [[-0.0872,  0.0286],
         [-0.0991,  0.0501],
         [-0.0999,  0.0633],
         [-0.0983,  0.0489],
         [-0.0514,  0.1098],
         [-0.0754,  0.0693]]], grad_fn=<UnsafeViewBackward0>)

In [61]:
batch.shape

torch.Size([2, 6, 3])

# Extending single-head attention to multi-head attention

## Stacking multiple single-head attention layer

In [124]:
class MultiHeadAttentionWarper(nn.Module):
    def __init__(self,d_in,d_out,context_len, dropout, num_heads = 2, qkv_bias = False):
        super().__init__()
        self.heads = nn.ModuleList([
            CasualAttention(d_in,d_out,context_len,dropout,qkv_bias) for _ in range(num_heads)
        ])

    def forward(self,x):
        return torch.cat([head(x) for head in self.heads], dim = -1)

torch.manual_seed(123)

context_len = batch.shape[1]

d_in,d_out = batch.shape[-1],2

mha = MultiHeadAttentionWarper(d_in,d_out,context_len,dropout = 0.0, num_heads =2)
mha(batch)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)

In [117]:
print("HI")

HI


## Implementing multi-head attention with weight splits

In [151]:

import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_len, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired o/p dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head o/p
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_len, context_len), diagonal=1)
        )

    def forward(self, x):

        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)          # Shape: (b,num_tokens,d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a "num_heads" dim
        # Unroll last dim : (b,num_tokens,d_out) --> (b,num_tokens,num_heads,head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose : (b, num_tokens, num_heads, head_dim) --> (b,num_tokens,num_heads,head_dim)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)
        queries = queries.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # original mask truncated to the no:of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        att_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        att_weights = self.dropout(att_weights)

        # Shape : (b,num_tokens,num_heads,head_dim)
        context_vec = (att_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)                    # can also written as context_vec.reshape(b, num_tokens, self.d_out)

        context_vec = self.out_proj(context_vec)

        return context_vec


torch.manual_seed(123)

batch_size, context_len, d_in = batch.shape
d_out = 4

mha = MultiHeadAttention(d_in, d_out, context_len, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("Context_vecs.shape:", context_vecs.shape)


tensor([[[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]],

        [[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0178,  0.3221, -0.0763, -0.4225],
         [-0.0147,  0.3259, -0.0734, -0.3721],
         [-0.0116,  0.3138, -0.0708, -0.3624],
         [-0.0117,  0.2973, -0.0698, -0.3543],
         [-0.0132,  0.2990, -0.0689, -0.3490]]], grad_fn=<ViewBackward0>)
Context_vecs.shape: torch.Size([2, 6, 4])
